# Simultaneous Lyα-forest reconstruction of density and temperature

## A multi-task one-dimensional U-Net

A Lyα forest spectrum depends on more than density. The neutral-hydrogen abundance,
thermal broadening, and temperature-dependent recombination physics all shape the
transmitted flux. In this notebook, a single one-dimensional U-Net receives an
observed Lyα spectrum and simultaneously predicts

$$
F_{\rm obs}(x)
\quad\longrightarrow\quad
\left\{\Delta_b(x),\,T(x)\right\}.
$$

The model has one shared encoder–decoder backbone and two output heads: one for
gas overdensity and one for temperature.

> **Forward-model rule**  
> Every spectrum is generated from the simulated H I density and temperature using
> a continuous Voigt calculation, instrumental smoothing, and noise. The target
> gas density and temperature are not inserted into an approximate invertible
> formula.

We evaluate both reconstructed fields along individual sightlines, across seven
held-out simulation realizations, and through the recovered temperature–density
relation.

## Learning objectives

By the end of the notebook, a student should be able to:

1. explain why density and temperature inference from one flux field is ill-posed;
2. generate realistic paired Lyα spectra and field labels without leakage;
3. standardize two targets with very different physical scales;
4. construct a shared 1D U-Net with two task-specific output heads;
5. interpret a balanced multi-task loss;
6. validate density and temperature fields separately at matched resolution;
7. compare one-point distributions and scale-dependent power; and
8. recover and interpret the median IGM temperature–density relation.


## 1. Requirements and reproducibility

Required packages are `numpy`, `scipy`, `matplotlib`, and `torch`. The model is
deliberately compact and runs on CPU. Fixed random seeds make the dataset sampling,
mock noise, weight initialization, and mini-batch ordering reproducible.

In [ ]:
from pathlib import Path
import time

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import LogNorm
from matplotlib.patches import FancyArrowPatch, FancyBboxPatch
from scipy import ndimage, special

import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, TensorDataset

np.random.seed(7)
torch.manual_seed(7)
torch.use_deterministic_algorithms(True)
torch.set_num_threads(min(4, max(1, torch.get_num_threads())))
device = torch.device("cpu")

plt.rcParams.update({
    "figure.figsize": (11, 5),
    "figure.dpi": 120,
    "font.size": 11,
    "axes.titlesize": 13,
    "axes.labelsize": 11,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "legend.frameon": False,
})

BLACK = "black"
BLUE = "royalblue"
ORANGE = "darkorange"
PURPLE = "slateblue"
RED = "crimson"
GREY = "grey"

print(f"PyTorch {torch.__version__}; device={device}")
print(f"CPU threads used by PyTorch={torch.get_num_threads()}")

## 2. Configuration

The defaults are chosen for a laptop-scale teaching exercise. The observation and
field-comparison scales match the preceding inversion notebooks.

In [ ]:
# Data and cosmology
data_directory = Path("Sims/CMD_z=2_grid128")
box_size = 25.0              # h^-1 cMpc
box_redshift = 2.0
h_camels = 0.6711
Omega_m = 0.30

# Mock observation
signal_to_noise = 30.0
instrument_fwhm_kms = 50.0

# Neural-network architecture
number_of_encoder_decoder_layers = 2
convolution_kernel_size = 5       # use a positive odd integer
base_channels = 8

# Dataset and training
training_skewers_per_box = 96
validation_skewers_per_box = 64
test_skewers_per_box = 16
number_of_epochs = 100
batch_size = 64
learning_rate = 1.0e-3
density_loss_weight = 1.0
temperature_loss_weight = 1.0

# Evaluation
comparison_fwhm = 1.0        # h^-1 cMpc
tdr_delta_min = 0.10
tdr_delta_max = 2.00
tdr_number_of_bins = 20

## 3. Load the CAMELS fields and define the data split

The target baryon overdensity is

$$
\Delta_b(x)=\frac{\rho_b(x)}{\bar\rho_b},
$$

where $\bar\rho_b$ is measured from the full three-dimensional simulation box.
The temperature target is the mass-weighted gas-temperature field supplied by the
gridded CAMELS data. It should not be confused with an H I-weighted line temperature
from a dedicated spectral pipeline.

We split by complete simulation realization:

| Simulations | Role | Used for gradient updates? |
|---|---|:---:|
| 0–17 | training | yes |
| 18–19 | validation and model selection | no |
| 20–26 | final testing | never |

This prevents sightlines from the same test realization entering training.

In [ ]:
gas_path = data_directory / "Grids_Mgas_IllustrisTNG_CV_128_z=2.0.npy"
hi_path = data_directory / "Grids_HI_IllustrisTNG_CV_128_z=2.0.npy"
temperature_path = data_directory / "Grids_T_IllustrisTNG_CV_128_z=2.0.npy"

missing_paths = [
    path for path in (gas_path, hi_path, temperature_path) if not path.exists()
]
if missing_paths:
    missing_names = "\n".join(f"  - {path}" for path in missing_paths)
    raise FileNotFoundError(
        "The CAMELS grids are required. Missing files:\n" + missing_names
    )

gas_boxes = np.load(gas_path, mmap_mode="r")
hi_boxes = np.load(hi_path, mmap_mode="r")
temperature_boxes = np.load(temperature_path, mmap_mode="r")

assert gas_boxes.shape == hi_boxes.shape == temperature_boxes.shape
assert gas_boxes.shape[0] >= 27

number_of_cells = gas_boxes.shape[1]
cell_size = box_size / number_of_cells
x = (np.arange(number_of_cells) + 0.5) * cell_size

training_simulations = np.arange(0, 18)
validation_simulations = np.arange(18, 20)
test_simulations = np.arange(20, 27)

gas_means = np.array([
    gas_boxes[simulation].mean(dtype=np.float64)
    for simulation in range(gas_boxes.shape[0])
])

print(f"Grid shape: {gas_boxes.shape}")
print(f"Cell width: {cell_size:.3f} h^-1 cMpc")
print(f"Training / validation / test boxes: "
      f"{len(training_simulations)} / {len(validation_simulations)} / "
      f"{len(test_simulations)}")

## 4. H I and temperature forward model

For absorber cell $i$, the thermal Doppler parameter is

$$b_i=\sqrt{\frac{2k_{\rm B}T_i}{m_{\rm H}}}.$$

Every absorber contributes a Voigt profile to neighbouring spectral pixels. The
resulting optical depth is non-local:

$$\tau(v_j)=\sum_i\tau_i(v_j),\qquad F(v_j)=e^{-\tau(v_j)}.$$

The mock observation includes Hubble-flow mapping, thermal and natural broadening,
Gaussian instrumental smoothing, and Gaussian pixel noise. Signed line-of-sight
peculiar velocities are omitted because these grids do not supply a gas-velocity
field.

In [ ]:
# Physical constants in cgs units
MSUN_G = 1.98847e33
MPC_CM = 3.085677581e24
M_H_G = 1.6735575e-24
K_B = 1.380649e-16
C_CMS = 2.99792458e10
C_KMS = C_CMS / 1.0e5

# Ly-alpha atomic constants
LAMBDA_ALPHA_CM = 1215.67e-8
GAMMA_ALPHA = 6.262e8
I_ALPHA = 4.45e-18

H_z = 100.0 * h_camels * np.sqrt(
    Omega_m * (1.0 + box_redshift)**3 + (1.0 - Omega_m)
)
distance_from_centre_Mpc = (x - box_size / 2.0) / h_camels
absorber_redshift = (
    box_redshift + H_z * distance_from_centre_Mpc / C_KMS
)


def hi_grid_to_number_density(rho_hi_grid):
    '''Convert CMD H I density to physical H I number density in cm^-3.'''
    rho_comoving = rho_hi_grid * MSUN_G * h_camels**2 / MPC_CM**3
    rho_physical = rho_comoving * (1.0 + absorber_redshift)**3
    return rho_physical / M_H_G


def continuous_voigt_optical_depth(rho_hi_grid, temperature):
    '''Optical depth from cell-sampled H I and temperature fields.'''
    n_hi = hi_grid_to_number_density(rho_hi_grid)
    nu_alpha = C_CMS / LAMBDA_ALPHA_CM
    cell_width_cm = cell_size * MPC_CM / h_camels

    safe_temperature = np.clip(temperature, 10.0, None)
    b_cms = np.sqrt(2.0 * K_B * safe_temperature / M_H_G)
    damping = GAMMA_ALPHA * C_CMS / (4.0 * np.pi * nu_alpha * b_cms)

    velocity_offset = (
        C_CMS
        * (absorber_redshift[:, None] - absorber_redshift[None, :])
        / (1.0 + absorber_redshift[None, :])
    )
    profile = np.real(special.wofz(
        velocity_offset / b_cms[:, None] + 1j*damping[:, None]
    ))
    amplitude = (
        C_CMS * I_ALPHA * cell_width_cm * n_hi
        / (np.sqrt(np.pi) * b_cms * (1.0 + absorber_redshift))
    )
    return np.sum(amplitude[:, None] * profile, axis=0)


velocity_pixel_width = H_z * (cell_size / h_camels) / (1.0 + box_redshift)
instrument_sigma_pixels = (
    instrument_fwhm_kms
    / (2.0 * np.sqrt(2.0 * np.log(2.0)))
    / velocity_pixel_width
)
noise_sigma = 1.0 / signal_to_noise


def observe_skewer(rho_hi, temperature, noise_rng):
    optical_depth = continuous_voigt_optical_depth(rho_hi, temperature)
    intrinsic_flux = np.exp(-optical_depth)
    instrumental_flux = ndimage.gaussian_filter1d(
        intrinsic_flux, instrument_sigma_pixels, mode="wrap"
    )
    observed_flux = instrumental_flux + noise_rng.normal(
        0.0, noise_sigma, number_of_cells
    )
    return intrinsic_flux, instrumental_flux, observed_flux


print(f"Instrument FWHM: {instrument_fwhm_kms:.1f} km/s")
print(f"Instrument sigma: {instrument_sigma_pixels:.2f} pixels")
print(f"S/N: {signal_to_noise:.1f}; flux-noise sigma: {noise_sigma:.4f}")

## 5. Construct paired multi-task examples

Each training example contains

$$
\underbrace{F_{\rm obs}(x)}_{\text{one input channel}}
\quad\longrightarrow\quad
\underbrace{\left[ln\Delta_b(x),\ln T(x)\right]}_{\text{two target channels}}.
$$

Logarithms reduce dynamic range and enforce positive predictions after
exponentiation. The central sightline is included first in every test simulation,
providing a deterministic visual comparison.

In [ ]:
def choose_positions(number, rng, include_centre=False):
    selected = []
    centre = number_of_cells // 2
    centre_flat = centre*number_of_cells + centre
    if include_centre:
        selected.append((centre, centre))

    available = np.delete(np.arange(number_of_cells**2), centre_flat)
    random_flat = rng.choice(
        available, number-len(selected), replace=False
    )
    selected.extend(divmod(int(index), number_of_cells) for index in random_flat)
    return selected


def build_dataset(simulations, skewers_per_box, seed, include_centre=False):
    position_rng = np.random.default_rng(seed)
    noise_rng = np.random.default_rng(seed + 1)

    densities, temperatures = [], []
    intrinsic_fluxes, instrumental_fluxes, observed_fluxes = [], [], []
    simulation_labels = []

    for simulation in simulations:
        positions = choose_positions(
            skewers_per_box, position_rng, include_centre=include_centre
        )
        for y_index, z_index in positions:
            density = (
                gas_boxes[simulation, :, y_index, z_index].astype(float)
                / gas_means[simulation]
            )
            rho_hi = hi_boxes[
                simulation, :, y_index, z_index
            ].astype(float)
            temperature = temperature_boxes[
                simulation, :, y_index, z_index
            ].astype(float)

            intrinsic, instrumental, observed = observe_skewer(
                rho_hi, temperature, noise_rng
            )
            densities.append(density)
            temperatures.append(temperature)
            intrinsic_fluxes.append(intrinsic)
            instrumental_fluxes.append(instrumental)
            observed_fluxes.append(observed)
            simulation_labels.append(simulation)

    return {
        "density": np.asarray(densities, dtype=np.float32),
        "temperature": np.asarray(temperatures, dtype=np.float32),
        "intrinsic_flux": np.asarray(intrinsic_fluxes, dtype=np.float32),
        "instrumental_flux": np.asarray(instrumental_fluxes, dtype=np.float32),
        "observed_flux": np.asarray(observed_fluxes, dtype=np.float32),
        "simulation": np.asarray(simulation_labels),
    }

In [ ]:
generation_start = time.perf_counter()

train_data = build_dataset(
    training_simulations, training_skewers_per_box, seed=101
)
validation_data = build_dataset(
    validation_simulations, validation_skewers_per_box, seed=202
)
test_data = build_dataset(
    test_simulations, test_skewers_per_box,
    seed=303, include_centre=True
)

print("Training:  ", train_data["observed_flux"].shape)
print("Validation:", validation_data["observed_flux"].shape)
print("Test:      ", test_data["observed_flux"].shape)
print(f"Dataset generation time: {time.perf_counter()-generation_start:.1f} s")

## 6. Inspect one complete example

The upper panel shows the progression from the intrinsic Voigt spectrum to the
noisy observed spectrum. The lower panels show the two hidden targets. The observed
data are drawn as a continuous step trace, as a sampled spectrum would normally be
displayed.

In [ ]:
representative = 0
representative_simulation = int(test_data["simulation"][representative])

fig, axes = plt.subplots(
    3, 1, figsize=(11, 8.2), sharex=True, constrained_layout=True
)
axes[0].plot(x, test_data["intrinsic_flux"][representative],
             color=BLACK, lw=1.5, label="intrinsic Voigt flux")
axes[0].plot(x, test_data["instrumental_flux"][representative],
             color=BLUE, lw=1.8, label="after instrument")
axes[0].step(x, test_data["observed_flux"][representative], where="mid",
             color=ORANGE, lw=1.0, label="noisy observed spectrum")
axes[0].set_ylim(-0.12, 1.12)
axes[0].set_ylabel("transmitted flux")
axes[0].set_title(f"Held-out simulation {representative_simulation}")
axes[0].legend(ncol=3)

axes[1].semilogy(x, test_data["density"][representative],
                 color=BLACK, lw=2.0)
axes[1].set_ylabel(r"gas $\Delta_b$")

axes[2].semilogy(x, test_data["temperature"][representative],
                 color=PURPLE, lw=1.8)
axes[2].set_ylabel(r"temperature $T$ [K]")
axes[2].set_xlabel(r"distance $x$ [$h^{-1}$ cMpc]")
plt.show()

## 7. Standardize one input and two targets

The targets have different units and dynamic ranges. We therefore standardize
their logarithms independently using training-set statistics:

$$
\widetilde s_\rho=
\frac{\ln\Delta_b-\mu_\rho}{\sigma_\rho},
\qquad
\widetilde s_T=
\frac{\ln T-\mu_T}{\sigma_T}.
$$

Equal loss weights then refer to comparable standardized errors, rather than a
meaningless comparison between kelvin and dimensionless overdensity.

In [ ]:
flux_mean = float(train_data["observed_flux"].mean())
flux_std = float(train_data["observed_flux"].std())

train_log_density = np.log(np.clip(train_data["density"], 1.0e-4, None))
train_log_temperature = np.log(
    np.clip(train_data["temperature"], 10.0, None)
)
validation_log_density = np.log(
    np.clip(validation_data["density"], 1.0e-4, None)
)
validation_log_temperature = np.log(
    np.clip(validation_data["temperature"], 10.0, None)
)

density_mean = float(train_log_density.mean())
density_std = float(train_log_density.std())
temperature_mean = float(train_log_temperature.mean())
temperature_std = float(train_log_temperature.std())


def flux_tensor(flux):
    standardized = (flux-flux_mean) / flux_std
    return torch.tensor(standardized[:, None, :], dtype=torch.float32)


def target_tensor(log_density, log_temperature):
    density_channel = (log_density-density_mean) / density_std
    temperature_channel = (
        log_temperature-temperature_mean
    ) / temperature_std
    stacked = np.stack([density_channel, temperature_channel], axis=1)
    return torch.tensor(stacked, dtype=torch.float32)


training_set = TensorDataset(
    flux_tensor(train_data["observed_flux"]),
    target_tensor(train_log_density, train_log_temperature),
)
validation_set = TensorDataset(
    flux_tensor(validation_data["observed_flux"]),
    target_tensor(validation_log_density, validation_log_temperature),
)

loader_generator = torch.Generator().manual_seed(7)
training_loader = DataLoader(
    training_set, batch_size=batch_size, shuffle=True,
    generator=loader_generator, num_workers=0
)
validation_loader = DataLoader(
    validation_set, batch_size=2*batch_size,
    shuffle=False, num_workers=0
)

example_flux, example_targets = next(iter(training_loader))
print("Input batch: ", tuple(example_flux.shape))
print("Target batch:", tuple(example_targets.shape))
print("Target channel 0 = ln density; channel 1 = ln temperature")

## 8. Multi-task U-Net architecture

The model consists of:

1. a shared encoder that extracts local and progressively broader features;
2. a shared bottleneck;
3. a shared decoder with U-Net skip connections; and
4. two independent $1\times1$ convolutional heads.

The encoder/decoder depth, convolutional kernel size, and base channel count are
set in the configuration cell; the decoder depth always mirrors the encoder.
The density and temperature heads see the same reconstructed feature representation
but learn different linear combinations of those features. This is simpler and more
interpretable than building two entirely separate networks.

In [ ]:
class ConvBlock(nn.Module):
    def __init__(self, input_channels, output_channels, kernel_size):
        super().__init__()
        if kernel_size < 1 or kernel_size % 2 == 0:
            raise ValueError("kernel_size must be a positive odd integer.")

        padding = kernel_size // 2
        self.layers = nn.Sequential(
            nn.Conv1d(
                input_channels, output_channels, kernel_size,
                padding=padding, padding_mode="circular"
            ),
            nn.ReLU(),
            nn.Conv1d(
                output_channels, output_channels, kernel_size,
                padding=padding, padding_mode="circular"
            ),
            nn.ReLU(),
        )

    def forward(self, inputs):
        return self.layers(inputs)

In [ ]:
class MultiTaskUNet1D(nn.Module):
    def __init__(self, number_of_layers, kernel_size, base_channels):
        super().__init__()
        if number_of_layers < 1:
            raise ValueError("number_of_layers must be at least 1.")

        self.pool = nn.MaxPool1d(2)
        self.encoder_blocks = nn.ModuleList()

        input_channels = 1
        for layer_index in range(number_of_layers):
            output_channels = base_channels * 2**layer_index
            self.encoder_blocks.append(
                ConvBlock(input_channels, output_channels, kernel_size)
            )
            input_channels = output_channels

        bottleneck_channels = base_channels * 2**number_of_layers
        self.bottleneck = ConvBlock(
            input_channels, bottleneck_channels, kernel_size
        )

        self.decoder_blocks = nn.ModuleList()
        current_channels = bottleneck_channels
        for layer_index in reversed(range(number_of_layers)):
            skip_channels = base_channels * 2**layer_index
            self.decoder_blocks.append(
                ConvBlock(
                    current_channels + skip_channels,
                    skip_channels,
                    kernel_size,
                )
            )
            current_channels = skip_channels

        self.density_head = nn.Conv1d(current_channels, 1, kernel_size=1)
        self.temperature_head = nn.Conv1d(current_channels, 1, kernel_size=1)

    def forward(self, inputs):
        encoder_features = []
        values = inputs

        for block in self.encoder_blocks:
            values = block(values)
            encoder_features.append(values)
            values = self.pool(values)

        shared_features = self.bottleneck(values)
        for block, skip_feature in zip(
            self.decoder_blocks, reversed(encoder_features)
        ):
            shared_features = F.interpolate(
                shared_features,
                size=skip_feature.shape[-1],
                mode="linear",
                align_corners=False,
            )
            shared_features = block(
                torch.cat([shared_features, skip_feature], dim=1)
            )

        density = self.density_head(shared_features)
        temperature = self.temperature_head(shared_features)
        return torch.cat([density, temperature], dim=1)


def parameter_count(model):
    return sum(parameter.numel() for parameter in model.parameters()
               if parameter.requires_grad)


torch.manual_seed(11)
model = MultiTaskUNet1D(
    number_of_layers=number_of_encoder_decoder_layers,
    kernel_size=convolution_kernel_size,
    base_channels=base_channels,
).to(device)
with torch.no_grad():
    example_prediction = model(example_flux[:2].to(device))

print(model)
print("Input shape: ", tuple(example_flux[:2].shape))
print("Output shape:", tuple(example_prediction.shape))
print(f"Trainable parameters: {parameter_count(model):,}")

In [ ]:
def architecture_box(ax, centre, text, colour, width=1.55, height=0.72):
    x_centre, y_centre = centre
    patch = FancyBboxPatch(
        (x_centre-width/2, y_centre-height/2), width, height,
        boxstyle="round,pad=0.04", facecolor=colour,
        edgecolor=BLACK, linewidth=1.0
    )
    ax.add_patch(patch)
    ax.text(x_centre, y_centre, text, ha="center", va="center", fontsize=9.5)


input_length = example_flux.shape[-1]
bottleneck_length = input_length // 2**number_of_encoder_decoder_layers
bottleneck_channels = base_channels * 2**number_of_encoder_decoder_layers

fig, ax = plt.subplots(figsize=(13.5, 4.0), constrained_layout=True)
nodes = [
    ((0.8, 1.4), f"Flux\n{input_length} x 1", "aliceblue"),
    (
        (3.2, 1.4),
        f"{number_of_encoder_decoder_layers} encoder blocks\n"
        f"{input_length} x {base_channels} to "
        f"{2*bottleneck_length} x {bottleneck_channels//2}",
        "lightblue",
    ),
    (
        (5.9, 1.4),
        f"Bottleneck\n{bottleneck_length} x {bottleneck_channels}",
        "lavender",
    ),
    (
        (8.6, 1.4),
        f"{number_of_encoder_decoder_layers} decoder blocks\n"
        f"{bottleneck_length} to {input_length} pixels",
        "bisque",
    ),
]
for centre, label, colour in nodes:
    architecture_box(ax, centre, label, colour)
for first, second in zip(nodes[:-1], nodes[1:]):
    ax.add_patch(FancyArrowPatch(
        (first[0][0]+0.78, 1.4), (second[0][0]-0.78, 1.4),
        arrowstyle="-|>", mutation_scale=12, color=GREY, linewidth=1.3
    ))

architecture_box(
    ax, (11.3, 2.15), f"Density head\n{input_length} x 1", "honeydew"
)
architecture_box(
    ax, (11.3, 0.65), f"Temperature head\n{input_length} x 1", "mistyrose"
)
ax.add_patch(FancyArrowPatch((9.4, 1.4), (10.5, 2.15),
                            arrowstyle="-|>", mutation_scale=12,
                            color=BLUE, linewidth=1.5))
ax.add_patch(FancyArrowPatch((9.4, 1.4), (10.5, 0.65),
                            arrowstyle="-|>", mutation_scale=12,
                            color=RED, linewidth=1.5))
ax.add_patch(FancyArrowPatch((3.2, 1.82), (8.6, 1.82),
                            connectionstyle="arc3,rad=-0.14",
                            arrowstyle="-|>", mutation_scale=11,
                            color=ORANGE, linewidth=1.6))
ax.text(5.9, 3.0, "U-Net skip connections", color=ORANGE,
        ha="center", fontweight="bold")
ax.set_xlim(0, 12.3)
ax.set_ylim(0.0, 3.3)
ax.axis("off")
ax.set_title("Shared feature reconstruction with two task-specific heads")
plt.show()

## 9. Multi-task loss

The total loss is the weighted mean of density and temperature losses:

$$
\mathcal L=
\frac{w_\rho\mathcal L_\rho+w_T\mathcal L_T}{w_\rho+w_T},
$$

where each component is MSE in its independently standardized log field. With
$w_\rho=w_T=1$, the two tasks contribute equally in standardized units.

Separate validation losses remain essential: a decreasing total loss can conceal
improvement in one task and degradation in the other.

In [ ]:
def loss_components(prediction, target):
    density_loss = F.mse_loss(prediction[:, 0], target[:, 0])
    temperature_loss = F.mse_loss(prediction[:, 1], target[:, 1])
    total_loss = (
        density_loss_weight*density_loss
        + temperature_loss_weight*temperature_loss
    ) / (density_loss_weight + temperature_loss_weight)
    return total_loss, density_loss, temperature_loss


def train_model(model):
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    history = {
        "train_total": [], "validation_total": [],
        "validation_density": [], "validation_temperature": [],
    }
    best_validation = np.inf
    best_state = None
    start = time.perf_counter()

    for epoch in range(1, number_of_epochs+1):
        model.train()
        training_sum = 0.0
        for input_batch, target_batch in training_loader:
            input_batch = input_batch.to(device)
            target_batch = target_batch.to(device)

            optimizer.zero_grad()
            total_loss, _, _ = loss_components(
                model(input_batch), target_batch
            )
            total_loss.backward()
            optimizer.step()
            training_sum += total_loss.item()*input_batch.size(0)

        model.eval()
        validation_sums = np.zeros(3)
        with torch.no_grad():
            for input_batch, target_batch in validation_loader:
                input_batch = input_batch.to(device)
                target_batch = target_batch.to(device)
                losses = loss_components(model(input_batch), target_batch)
                validation_sums += np.array([
                    loss.item()*input_batch.size(0) for loss in losses
                ])

        train_total = training_sum / len(training_set)
        validation_total, validation_density, validation_temperature = (
            validation_sums / len(validation_set)
        )
        history["train_total"].append(train_total)
        history["validation_total"].append(validation_total)
        history["validation_density"].append(validation_density)
        history["validation_temperature"].append(validation_temperature)

        if validation_total < best_validation:
            best_validation = validation_total
            best_state = {
                name: value.detach().cpu().clone()
                for name, value in model.state_dict().items()
            }

        if epoch == 1 or epoch % 5 == 0:
            print(
                f"epoch {epoch:2d}/{number_of_epochs}: "
                f"train={train_total:.4f}, val={validation_total:.4f}, "
                f"density={validation_density:.4f}, "
                f"temperature={validation_temperature:.4f}"
            )

    model.load_state_dict(best_state)
    print(f"Retained best validation model; CPU time={time.perf_counter()-start:.1f} s")
    return history


history = train_model(model)

In [ ]:
epochs = np.arange(1, number_of_epochs+1)
fig, axes = plt.subplots(1, 2, figsize=(11, 4.3), constrained_layout=True)

axes[0].semilogy(epochs, history["train_total"], color=BLUE, lw=1.8,
                 label="training total")
axes[0].semilogy(epochs, history["validation_total"], color=ORANGE, lw=2,
                 label="validation total")
best_epoch = int(np.argmin(history["validation_total"])) + 1
axes[0].axvline(best_epoch, color=GREY, ls=":",
                label=f"selected epoch {best_epoch}")
axes[0].set(xlabel="epoch", ylabel="standardized MSE",
            title="Total multi-task loss")
axes[0].legend()
axes[0].grid(alpha=0.2)

axes[1].semilogy(epochs, history["validation_density"], color=BLUE, lw=2,
                 label="density validation")
axes[1].semilogy(epochs, history["validation_temperature"], color=RED, lw=2,
                 label="temperature validation")
axes[1].set(xlabel="epoch", ylabel="standardized MSE",
            title="Task-specific validation losses")
axes[1].legend()
axes[1].grid(alpha=0.2)
plt.show()

## 10. Predict the held-out fields

The two output channels are independently unstandardized and exponentiated:

$$
\Delta_b^{\rm pred}=e^{\mu_\rho+\sigma_\rho\widetilde s_\rho},
\qquad
T^{\rm pred}=e^{\mu_T+\sigma_T\widetilde s_T}.
$$

Broad numerical bounds prevent overflow but do not impose the diffuse IGM equation
of state on either prediction.

In [ ]:
def predict_fields(model, observed_flux):
    inputs = flux_tensor(observed_flux).to(device)
    model.eval()
    with torch.no_grad():
        standardized = model(inputs).cpu().numpy()

    log_density = density_mean + density_std*standardized[:, 0]
    log_temperature = (
        temperature_mean + temperature_std*standardized[:, 1]
    )
    log_density = np.clip(log_density, np.log(1.0e-4), np.log(1.0e3))
    log_temperature = np.clip(
        log_temperature, np.log(10.0), np.log(1.0e9)
    )
    return np.exp(log_density), np.exp(log_temperature)


predicted_density, predicted_temperature = predict_fields(
    model, test_data["observed_flux"]
)
print("Predicted density shape:    ", predicted_density.shape)
print("Predicted temperature shape:", predicted_temperature.shape)

## 11. Visual comparison along one sightline

The density and temperature panels use logarithmic axes because both fields span
large dynamic ranges. Agreement in position and amplitude should be assessed
separately: a model may locate a structure correctly while smoothing its peak.

In [ ]:
fig, axes = plt.subplots(
    3, 1, figsize=(11, 8.4), sharex=True, constrained_layout=True
)
axes[0].step(x, test_data["observed_flux"][representative], where="mid",
             color=BLACK, lw=1.0, label="observed noisy spectrum")
axes[0].plot(x, test_data["instrumental_flux"][representative],
             color=BLUE, lw=1.7, label="noise-free instrument flux")
axes[0].set_ylim(-0.12, 1.12)
axes[0].set_ylabel("transmitted flux")
axes[0].set_title(f"Held-out simulation {representative_simulation}")
axes[0].legend(ncol=2)

axes[1].semilogy(x, test_data["density"][representative],
                 color=BLACK, lw=2.3, label="true density")
axes[1].semilogy(x, predicted_density[representative],
                 color=ORANGE, lw=2.0, label="predicted density")
axes[1].set_ylabel(r"gas $\Delta_b$")
axes[1].legend()

axes[2].semilogy(x, test_data["temperature"][representative],
                 color=BLACK, lw=2.1, label="true temperature")
axes[2].semilogy(x, predicted_temperature[representative],
                 color=RED, lw=1.9, label="predicted temperature")
axes[2].set_ylabel(r"$T$ [K]")
axes[2].set_xlabel(r"distance $x$ [$h^{-1}$ cMpc]")
axes[2].legend()
plt.show()

## 12. Visual ensemble across all seven test simulations

To avoid relying on a single attractive example, the following small multiples show
the central sightline from every held-out simulation. Each row is an independent
realization; density is shown on the left and temperature on the right.

In [ ]:
central_indices = [
    np.flatnonzero(test_data["simulation"] == simulation)[0]
    for simulation in test_simulations
]

fig, axes = plt.subplots(
    len(test_simulations), 2, figsize=(13, 14), sharex=True,
    constrained_layout=True
)
for row, (simulation, index) in enumerate(zip(test_simulations, central_indices)):
    axes[row, 0].semilogy(x, test_data["density"][index],
                         color=BLACK, lw=1.5)
    axes[row, 0].semilogy(x, predicted_density[index],
                         color=ORANGE, lw=1.3)
    axes[row, 0].set_ylabel(f"sim {simulation}\n" + r"$\Delta_b$")

    axes[row, 1].semilogy(x, test_data["temperature"][index],
                         color=BLACK, lw=1.5)
    axes[row, 1].semilogy(x, predicted_temperature[index],
                         color=RED, lw=1.3)
    axes[row, 1].set_ylabel(r"$T$ [K]")

axes[0, 0].set_title("Gas overdensity: true (black), predicted (orange)")
axes[0, 1].set_title("Temperature: true (black), predicted (red)")
axes[-1, 0].set_xlabel(r"distance $x$ [$h^{-1}$ cMpc]")
axes[-1, 1].set_xlabel(r"distance $x$ [$h^{-1}$ cMpc]")
plt.show()

## 13. Matched-resolution field metrics

We smooth truth and prediction to a common $1\,h^{-1}{\rm cMpc}$ FWHM. Metrics are
first calculated per sightline, averaged over the 16 sightlines within each test
box, and finally summarized by the mean and standard deviation across simulations
20–26.

For either positive field $q$,

$$
{\rm RMSE}_{\log q}=
\sqrt{\left\langle
(\log_{10}q_{\rm pred}-\log_{10}q_{\rm true})^2
\right\rangle}.
$$

In [ ]:
comparison_sigma_pixels = (
    comparison_fwhm
    / (2.0*np.sqrt(2.0*np.log(2.0)))
    / cell_size
)


def smooth_fields(fields):
    return ndimage.gaussian_filter1d(
        fields, comparison_sigma_pixels, axis=1, mode="wrap"
    )


def field_metrics_per_skewer(truth, prediction):
    log_truth = np.log10(np.clip(truth, 1.0e-8, None))
    log_prediction = np.log10(np.clip(prediction, 1.0e-8, None))
    difference = log_prediction-log_truth
    return {
        "RMSE [dex]": np.sqrt(np.mean(difference**2, axis=1)),
        "bias [dex]": np.mean(difference, axis=1),
        "correlation": np.array([
            np.corrcoef(true_row, prediction_row)[0, 1]
            for true_row, prediction_row in zip(log_truth, log_prediction)
        ]),
    }


def average_within_each_test_simulation(values):
    return np.array([
        np.mean(values[test_data["simulation"] == simulation], axis=0)
        for simulation in test_simulations
    ])


density_truth_matched = smooth_fields(test_data["density"])
density_prediction_matched = smooth_fields(predicted_density)
temperature_truth_matched = smooth_fields(test_data["temperature"])
temperature_prediction_matched = smooth_fields(predicted_temperature)

metric_samples = {}
for field_name, truth, prediction in [
    ("Density", density_truth_matched, density_prediction_matched),
    ("Temperature", temperature_truth_matched, temperature_prediction_matched),
]:
    per_skewer = field_metrics_per_skewer(truth, prediction)
    metric_samples[field_name] = {
        metric: average_within_each_test_simulation(values)
        for metric, values in per_skewer.items()
    }

print("Mean +/- standard deviation across seven held-out simulations")
print(f"{'Field':14s} {'RMSE [dex]':>20s} {'bias [dex]':>20s} "
      f"{'correlation':>20s}")
for field_name, samples in metric_samples.items():
    print(
        f"{field_name:14s} "
        f"{samples['RMSE [dex]'].mean():7.3f} +/- "
        f"{samples['RMSE [dex]'].std(ddof=1):.3f} "
        f"{samples['bias [dex]'].mean():+7.3f} +/- "
        f"{samples['bias [dex]'].std(ddof=1):.3f} "
        f"{samples['correlation'].mean():7.3f} +/- "
        f"{samples['correlation'].std(ddof=1):.3f}"
    )

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10.5, 4.3), constrained_layout=True)
field_names = list(metric_samples)
colours = [ORANGE, RED]

rmse_mean = [metric_samples[name]["RMSE [dex]"].mean() for name in field_names]
rmse_std = [metric_samples[name]["RMSE [dex]"].std(ddof=1) for name in field_names]
correlation_mean = [
    metric_samples[name]["correlation"].mean() for name in field_names
]
correlation_std = [
    metric_samples[name]["correlation"].std(ddof=1) for name in field_names
]

axes[0].bar(field_names, rmse_mean, yerr=rmse_std,
            color=colours, capsize=4)
axes[0].set_ylabel("log-field RMSE [dex]")
axes[0].set_title("Lower is better")
axes[0].grid(axis="y", alpha=0.2)

axes[1].bar(field_names, correlation_mean, yerr=correlation_std,
            color=colours, capsize=4)
axes[1].set_ylim(max(0.0, min(correlation_mean)-0.15), 1.0)
axes[1].set_ylabel("log-field correlation")
axes[1].set_title("Higher is better")
axes[1].grid(axis="y", alpha=0.2)
fig.suptitle("Matched-resolution field accuracy")
plt.show()

## 14. Cell-by-cell statistical comparison

Hexagonal density maps show whether predictions are unbiased across the full
dynamic range. A narrow diagonal distribution indicates accurate recovery; a
horizontal compression indicates regression toward the mean.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10.5, 4.8), constrained_layout=True)
comparisons = [
    (density_truth_matched, density_prediction_matched,
     r"$\log_{10}\Delta_b^{\rm true}$",
     r"$\log_{10}\Delta_b^{\rm pred}$", "Density"),
    (temperature_truth_matched, temperature_prediction_matched,
     r"$\log_{10}T^{\rm true}$",
     r"$\log_{10}T^{\rm pred}$", "Temperature"),
]

for ax, (truth, prediction, x_label, y_label, title) in zip(axes, comparisons):
    x_values = np.log10(np.clip(truth.ravel(), 1.0e-8, None))
    y_values = np.log10(np.clip(prediction.ravel(), 1.0e-8, None))
    limits = [min(x_values.min(), y_values.min()),
              max(x_values.max(), y_values.max())]
    image = ax.hexbin(x_values, y_values, gridsize=55, bins="log",
                      mincnt=1, cmap="magma")
    ax.plot(limits, limits, color="turquoise", ls="--", lw=1.8,
            label="perfect recovery")
    ax.set(xlabel=x_label, ylabel=y_label, title=title,
           xlim=limits, ylim=limits)
    ax.legend()
    fig.colorbar(image, ax=ax, pad=0.02, label="number of pixels")
plt.show()

## 15. PDFs and scale-dependent power

Field metrics are complemented by four summary statistics:

- PDF of $\log_{10}\Delta_b$;
- PDF of $\log_{10}T$;
- power spectrum of the density contrast;
- power spectrum of log-temperature fluctuations.

Each curve is averaged over sightlines within a simulation. Error bars are the
standard deviation across the seven test simulations.

In [ ]:
density_pdf_edges = np.linspace(-2.0, 2.5, 34)
temperature_pdf_edges = np.linspace(2.0, 8.0, 34)
density_pdf_centres = 0.5*(density_pdf_edges[:-1]+density_pdf_edges[1:])
temperature_pdf_centres = 0.5*(
    temperature_pdf_edges[:-1]+temperature_pdf_edges[1:]
)


def pdf_per_skewer(field, edges):
    return np.asarray([
        np.histogram(np.log10(np.clip(row, 1.0e-8, None)),
                     bins=edges, density=True)[0]
        for row in field
    ])


def power_per_skewer(field, logarithmic=False):
    if logarithmic:
        fluctuations = np.log(field)-np.log(field).mean(axis=1, keepdims=True)
    else:
        fluctuations = field/field.mean(axis=1, keepdims=True)-1.0
    transform = cell_size*np.fft.rfft(fluctuations, axis=1)
    k = 2.0*np.pi*np.fft.rfftfreq(number_of_cells, d=cell_size)
    power = np.abs(transform)**2/box_size
    return k[1:], power[:, 1:]


def bin_modes(k, values, number_of_bins=10):
    edges = np.logspace(np.log10(k.min()), np.log10(k.max()),
                        number_of_bins+1)
    centres = np.sqrt(edges[:-1]*edges[1:])
    binned = np.full((values.shape[0], number_of_bins), np.nan)
    for index in range(number_of_bins):
        in_bin = (k >= edges[index]) & (k < edges[index+1])
        if np.any(in_bin):
            binned[:, index] = values[:, in_bin].mean(axis=1)
    return centres, binned

In [ ]:
summary_fields = {
    "True": (density_truth_matched, temperature_truth_matched),
    "Predicted": (density_prediction_matched, temperature_prediction_matched),
}
summaries = {}

for label, (density, temperature) in summary_fields.items():
    density_pdf = pdf_per_skewer(density, density_pdf_edges)
    temperature_pdf = pdf_per_skewer(temperature, temperature_pdf_edges)

    density_k_raw, density_power_raw = power_per_skewer(density)
    density_k, density_power = bin_modes(density_k_raw, density_power_raw)

    temperature_k_raw, temperature_power_raw = power_per_skewer(
        temperature, logarithmic=True
    )
    temperature_k, temperature_power = bin_modes(
        temperature_k_raw, temperature_power_raw
    )

    summaries[label] = {
        "density_pdf": average_within_each_test_simulation(density_pdf),
        "temperature_pdf": average_within_each_test_simulation(temperature_pdf),
        "density_power": average_within_each_test_simulation(density_power),
        "temperature_power": average_within_each_test_simulation(
            temperature_power
        ),
    }

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8.5), constrained_layout=True)
style = {
    "True": (BLACK, "o"),
    "Predicted": (ORANGE, "s"),
}

plot_information = [
    (axes[0, 0], "density_pdf", density_pdf_centres,
     r"$\log_{10}\Delta_b$", "probability density", "Density PDF", False),
    (axes[0, 1], "temperature_pdf", temperature_pdf_centres,
     r"$\log_{10}T$ [K]", "probability density", "Temperature PDF", False),
    (axes[1, 0], "density_power", density_k,
     r"$k$ [$h$ cMpc$^{-1}$]", r"$P_{\delta_b}(k)$",
     "Density power", True),
    (axes[1, 1], "temperature_power", temperature_k,
     r"$k$ [$h$ cMpc$^{-1}$]", r"$P_{\ln T}(k)$",
     "Log-temperature power", True),
]

for ax, statistic, coordinates, x_label, y_label, title, log_x in plot_information:
    for label, statistics in summaries.items():
        values = statistics[statistic]
        mean = np.nanmean(values, axis=0)
        std = np.nanstd(values, axis=0, ddof=1)
        valid = np.isfinite(mean) & (mean > 0.0)
        colour, marker = style[label]
        ax.errorbar(coordinates[valid], mean[valid], yerr=std[valid],
                    color=colour, marker=marker, ms=4, lw=1.5,
                    capsize=2, errorevery=2, label=label)
    if log_x:
        ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set(xlabel=x_label, ylabel=y_label, title=title)
    ax.grid(alpha=0.18, which="both")
    ax.legend()

fig.suptitle("Mean and standard deviation across test simulations 20–26")
plt.show()

## 16. True and predicted temperature–density phase diagrams

The temperature–density relation is a joint-field diagnostic. A model may recover
the marginal density and temperature PDFs while pairing the wrong temperature with
a given density.

We compare the distributions of $(\Delta_b,T)$ using identical logarithmic bins and
a shared colour normalization. The white curves show median temperature in density
bins. Both axes use the model's own paired fields: true $T$ versus true $\Delta_b$,
and predicted $T$ versus predicted $\Delta_b$.

In [ ]:
phase_density_edges = np.logspace(-3.0, 3.0, 90)
phase_temperature_edges = np.logspace(2.0, 8.0, 90)
tdr_edges = np.logspace(np.log10(0.05), np.log10(10.0),
                        tdr_number_of_bins+1)
tdr_centres = np.sqrt(tdr_edges[:-1]*tdr_edges[1:])


def phase_histogram(density, temperature):
    histogram, _, _ = np.histogram2d(
        density.ravel(), temperature.ravel(),
        bins=(phase_density_edges, phase_temperature_edges)
    )
    return histogram


def median_temperature_relation(density, temperature):
    density_flat = density.ravel()
    temperature_flat = temperature.ravel()
    medians = np.full(tdr_number_of_bins, np.nan)
    bin_number = np.digitize(density_flat, tdr_edges)-1
    for index in range(tdr_number_of_bins):
        values = temperature_flat[bin_number == index]
        if values.size >= 20:
            medians[index] = np.median(values)
    return medians


true_phase_histogram = phase_histogram(
    test_data["density"], test_data["temperature"]
)
predicted_phase_histogram = phase_histogram(
    predicted_density, predicted_temperature
)
shared_maximum = max(
    true_phase_histogram.max(), predicted_phase_histogram.max()
)
shared_norm = LogNorm(vmin=1.0, vmax=shared_maximum)

true_median_tdr = median_temperature_relation(
    test_data["density"], test_data["temperature"]
)
predicted_median_tdr = median_temperature_relation(
    predicted_density, predicted_temperature
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5.3),
                         sharex=True, sharey=True, constrained_layout=True)
images = []
for ax, histogram, medians, title in [
    (axes[0], true_phase_histogram, true_median_tdr, "True fields"),
    (axes[1], predicted_phase_histogram, predicted_median_tdr,
     "Multi-task U-Net fields"),
]:
    image = ax.pcolormesh(
        phase_density_edges, phase_temperature_edges,
        histogram.T, norm=shared_norm, cmap="magma", shading="auto"
    )
    images.append(image)
    valid = np.isfinite(medians)
    ax.plot(tdr_centres[valid], medians[valid], color="white",
            marker="o", ms=3.5, lw=1.6,
            markeredgecolor=BLACK, markeredgewidth=0.35,
            label="median temperature")
    ax.axvspan(tdr_delta_min, tdr_delta_max,
               color="turquoise", alpha=0.09, label="fit range")
    ax.set(xscale="log", yscale="log",
           xlabel=r"gas overdensity $\Delta_b$", title=title)
    ax.legend(loc="lower right", fontsize=8)

axes[0].set_ylabel(r"temperature $T$ [K]")
colourbar = fig.colorbar(images[-1], ax=axes, pad=0.02)
colourbar.set_label("number of test pixels")
fig.suptitle("Temperature–density phase distribution")
plt.show()

## 17. Median temperature–density relation and fitted parameters

For each test simulation, its 16 sightlines are combined and median temperature is
calculated in logarithmic density bins. We fit

$$
\log_{10}T=\log_{10}T_0+(\gamma-1)\log_{10}\Delta_b
$$

over $0.1\leq\Delta_b\leq3$. The mean curve and error bars below are the mean and
standard deviation across the seven independent simulations.

In [ ]:
def tdr_by_simulation(density, temperature):
    relations, T0_values, gamma_values = [], [], []
    for simulation in test_simulations:
        select = test_data["simulation"] == simulation
        relation = median_temperature_relation(
            density[select], temperature[select]
        )
        fit = (
            np.isfinite(relation)
            & (tdr_centres >= tdr_delta_min)
            & (tdr_centres <= tdr_delta_max)
        )
        slope, intercept = np.polyfit(
            np.log10(tdr_centres[fit]), np.log10(relation[fit]), 1
        )
        relations.append(relation)
        T0_values.append(10.0**intercept)
        gamma_values.append(1.0+slope)
    return (
        np.asarray(relations),
        np.asarray(T0_values),
        np.asarray(gamma_values),
    )


true_relations, true_T0, true_gamma = tdr_by_simulation(
    test_data["density"], test_data["temperature"]
)
predicted_relations, predicted_T0, predicted_gamma = tdr_by_simulation(
    predicted_density, predicted_temperature
)

print("Temperature-density fits: mean +/- simulation standard deviation")
print(f"True:      T0={true_T0.mean():.0f} +/- {true_T0.std(ddof=1):.0f} K, "
      f"gamma={true_gamma.mean():.3f} +/- {true_gamma.std(ddof=1):.3f}")
print(f"Predicted: T0={predicted_T0.mean():.0f} +/- "
      f"{predicted_T0.std(ddof=1):.0f} K, "
      f"gamma={predicted_gamma.mean():.3f} +/- "
      f"{predicted_gamma.std(ddof=1):.3f}")

In [ ]:
fig, axes = plt.subplots(
    1, 3, figsize=(14, 4.8), constrained_layout=True,
    gridspec_kw={"width_ratios": [1.7, 1.0, 1.0]}
)

for relations, label, colour, marker in [
    (true_relations, "true", BLACK, "o"),
    (predicted_relations, "predicted", ORANGE, "s"),
]:
    available = np.sum(np.isfinite(relations), axis=0) >= 2
    mean = np.nanmean(relations[:, available], axis=0)
    std = np.nanstd(relations[:, available], axis=0, ddof=1)
    valid = np.isfinite(mean)
    available_centres = tdr_centres[available]
    axes[0].errorbar(available_centres[valid], mean[valid], yerr=std[valid],
                     color=colour, marker=marker, ms=4, lw=1.7,
                     capsize=2, errorevery=2, label=label)

axes[0].axvspan(tdr_delta_min, tdr_delta_max,
                color="turquoise", alpha=0.10, label="fitted range")
axes[0].set(xscale="log", yscale="log",
            xlabel=r"gas overdensity $\Delta_b$",
            ylabel=r"median temperature $T$ [K]",
            title="Median temperature–density relation")
axes[0].legend()
axes[0].grid(alpha=0.18, which="both")

parameter_labels = ["True", "Predicted"]
x_positions = np.arange(2)

axes[1].bar(
    x_positions, [true_T0.mean()/1000.0, predicted_T0.mean()/1000.0],
    yerr=[true_T0.std(ddof=1)/1000.0,
          predicted_T0.std(ddof=1)/1000.0],
    color=[BLACK, ORANGE], capsize=4
)
axes[1].set_xticks(x_positions, parameter_labels)
axes[1].set_ylabel(r"$T_0$ [kK]")
axes[1].set_title("Thermal normalization")
axes[1].grid(axis="y", alpha=0.18)

axes[2].bar(
    x_positions, [true_gamma.mean(), predicted_gamma.mean()],
    yerr=[true_gamma.std(ddof=1), predicted_gamma.std(ddof=1)],
    color=[GREY, RED], capsize=4
)
axes[2].set_xticks(x_positions, parameter_labels)
axes[2].set_ylabel(r"$\gamma$")
axes[2].set_title("Thermal slope")
axes[2].grid(axis="y", alpha=0.18)
plt.show()

### Interpreting the recovered relation

Three comparisons should be kept distinct:

1. **Phase occupancy:** does the prediction reproduce the cool diffuse sequence and
   the hot shock-heated population?
2. **Median relation:** does the typical temperature at fixed predicted density
   agree with the truth?
3. **Thermal parameters:** are $T_0$ and $\gamma$ unbiased across independent boxes?

MSE-trained networks often compress rare extremes toward the conditional mean.
Consequently, the predicted phase diagram may appear narrower and smoother than
the true distribution even when the median $T$–$\Delta_b$ relation is accurate.

## 18. What the experiment establishes

- A single flux field contains statistical information about both density and
  temperature because both influence H I absorption and line broadening.
- The two fields are not uniquely identifiable pixel by pixel. The network combines
  the spectrum with correlations learned from the training simulations.
- A shared backbone exploits features useful to both tasks, while separate heads
  allow task-specific outputs.
- Density and temperature must be validated independently; good density recovery
  does not imply good temperature recovery.
- Recovering both marginal PDFs is insufficient. The joint temperature–density
  phase distribution provides a stronger physical test.
- Predictions are conditional on the simulation family, forward physics, noise,
  resolution, targets, and loss function represented during training.

## 19. Assumptions and limitations

- The temperature label is the CAMELS mass-weighted gridded gas temperature.
- Hubble flow, thermal and natural broadening, instrumental smoothing, and Gaussian
  pixel noise are included.
- Signed gas peculiar velocities are omitted because the corresponding grid is not
  supplied.
- Continuum uncertainty, metal contamination, correlated noise, and spatially
  varying resolution are omitted.
- Training, validation, and testing use one hydrodynamical family and one observing
  configuration.
- The network returns deterministic point estimates, not posterior uncertainties.
- The recovered $T_0$ and $\gamma$ describe the median predicted phase relation;
  they are not direct independent measurements from each flux pixel.

## Final take-away

> A multi-task U-Net can reconstruct correlated density and temperature fields from
> a Lyα spectrum by combining shared spectral features with a simulation-learned
> prior. Its success must be judged not only from attractive sightline plots, but
> from held-out field metrics, marginal statistics, and the joint recovered
> temperature–density relation.